In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from transformers import AutoModel, AutoTokenizer

**Reward Model Architecture**

In [ ]:
class RewardModel(nn.Module):
    def __init__(self, base_model_name, hidden_size=768):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(base_model_name)
        
        self.reward_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )
    
    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        
        if hasattr(outputs, "pooler_output"):
            hidden = outputs.pooler_output
        else:
            hidden = (outputs.last_hidden_state * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1, keepdim=True)
        
        reward = self.reward_head(hidden)
        return reward.squeeze(-1)

In [4]:
def train_reward_model(reward_model, preference_data, num_epochs=3):
    optimizer = optim.AdamW(reward_model.parameters(), lr=1e-5)
    
    for epoch in range(num_epochs):
        total_loss = 0
        
        for batch in preference_data:
            chosen_ids = batch['chosen_input_ids']
            rejected_ids = batch['rejected_input_ids']
            chosen_mask = batch['chosen_attention_mask']
            rejected_mask = batch['rejected_attention_mask']
            
            chosen_reward = reward_model(chosen_ids, chosen_mask)
            rejected_reward = reward_model(rejected_ids, rejected_mask)
            
            loss = -F.logsigmoid(chosen_reward - rejected_reward).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(preference_data):.4f}")

In [5]:
raw_preference_data = [
    {
        "prompt": "What is the capital of France?",
        "chosen": "The capital of France is Paris.",
        "rejected": "I think the capital is London or maybe Berlin."
    },
    {
        "prompt": "Write a Python loop to print numbers 1 to 3.",
        "chosen": "for i in range(1, 4):\n    print(i)",
        "rejected": "print 1, 2, 3"
    },
    {
        "prompt": "How do you boil an egg?",
        "chosen": "Place the egg in boiling water for 6-9 minutes depending on how runny you want the yolk.",
        "rejected": "Just put the egg in the microwave for 5 minutes."
    },
    {
        "prompt": "Is Python statically or dynamically typed?",
        "chosen": "Python is a dynamically typed programming language.",
        "rejected": "Python is statically typed, meaning you must declare variable types."
    }
]

In [7]:
model_name = "distilbert-base-uncased"
print(f"Loading Tokenizer and Model ({model_name})...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = RewardModel(base_model_name=model_name, hidden_size=768)

Loading Tokenizer and Model (distilbert-base-uncased)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
processed_preference_data = []

for item in raw_preference_data:
    chosen_tokens = tokenizer(item["chosen"], padding="max_length", truncation=True, max_length=32, return_tensors="pt")
    rejected_tokens = tokenizer(item["rejected"], padding="max_length", truncation=True, max_length=32, return_tensors="pt")
        
    processed_preference_data.append({
        'chosen_input_ids': chosen_tokens['input_ids'],
        'chosen_attention_mask': chosen_tokens['attention_mask'],
        'rejected_input_ids': rejected_tokens['input_ids'],
        'rejected_attention_mask': rejected_tokens['attention_mask']
    })

In [9]:
train_reward_model(model, processed_preference_data, num_epochs=5)

Epoch 1, Loss: 0.6990
Epoch 2, Loss: 0.6370
Epoch 3, Loss: 0.5894
Epoch 4, Loss: 0.5421
Epoch 5, Loss: 0.4920
